In [ ]:
import pandas as pd
import networkx as nx

In [ ]:
df = pd.read_csv('export_api_influencers.csv')

print("Dimensiones originales:", df.shape)
display(df.head())

In [ ]:

df_limpio = df.dropna(subset=['Cuentas_Que_Sigue']).copy()

df_limpio['Usuario'] = df_limpio['Usuario'].str.upper().str.strip()

df_limpio = df_limpio.drop_duplicates(subset=['Usuario'])

print("Registros originales:", len(df))
print("Registros limpios:", len(df_limpio))

display(df_limpio.head())

In [ ]:

df_limpio['Cuentas_Que_Sigue'] = df_limpio['Cuentas_Que_Sigue'].str.split(',')

lista_aristas = df_limpio[['Usuario', 'Cuentas_Que_Sigue']].explode('Cuentas_Que_Sigue')

lista_aristas = lista_aristas.rename(columns={
    'Usuario': 'Origen',
    'Cuentas_Que_Sigue': 'Destino'
})

display(lista_aristas.head(10))

In [ ]:

lista_aristas['Destino'] = lista_aristas['Destino'].str.upper().str.strip()

lista_aristas = lista_aristas.dropna(subset=['Destino'])
lista_aristas = lista_aristas[lista_aristas['Destino'] != '']

lista_aristas = lista_aristas[lista_aristas['Origen'] != lista_aristas['Destino']]

lista_aristas = lista_aristas.reset_index(drop=True)

print("total de aristas:", len(lista_aristas))
display(lista_aristas)

In [ ]:

G = nx.from_pandas_edgelist(
    lista_aristas,
    source='Origen',
    target='Destino',
    create_using=nx.DiGraph()
)

print("numero de nodos:", G.number_of_nodes())
print("numero de aristas:", G.number_of_edges())

In [ ]:

centralidad_entrada = nx.in_degree_centrality(G)

top_3_entrada = sorted(
    centralidad_entrada.items(),
    key=lambda elemento: elemento[1],
    reverse=True
)[:3]

print("top 3 usuarios por centralidad de entrada:")
for usuario, valor in top_3_entrada:
    print(f"{usuario}: {valor:.6f}")

In [ ]:

pagerank = nx.pagerank(G)

top_3_pagerank = sorted(
    pagerank.items(),
    key=lambda elemento: elemento[1],
    reverse=True
)[:3]

print("top 3 usuarios por pagerank:")
for usuario, valor in top_3_pagerank:
    print(f"{usuario}: {valor:.6f}")

In [ ]:

G_no_dirigido = G.to_undirected()

clustering = nx.clustering(G_no_dirigido)

print(f"Clustering de AMATEUR_1: {clustering['AMATEUR_1']:.6f}")
print(f"Clustering de GARNACHO: {clustering['GARNACHO']:.6f}")